<a href="https://colab.research.google.com/github/Frogs-are-cool/RPG-battle-simulator-project/blob/main/RPG_battle_simulator_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ipywidgets --quiet
import ipywidgets as widgets
from IPython.display import display, clear_output

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 34.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import json, os ,random

drive.mount('/content/drive')

SAVE_DIR = "/conetent/drive/MyDrive/rpg_save"
os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_FILE =  os.path.join(SAVE_DIR, "save.json")


ValueError: mount failed

In [ ]:
player = load_game() or new_player()
current_enemy = None

In [ ]:
def save_game(player):
  tmp = SAVE_FILE + ".tmp"
  with open(tmp, "w") as f:
    json.dump(player, f, indent=2)
  os.replace(tmp, SAVE_FILE)


def load_game():
  try:
    with open(SAVE_FILE) as f:
      return json.load(f)
  except (FileNotFoundError, json.JSONDecodeError):
    return None

def upgrade_save(player):
    if "inventory" not in player:
        player["inventory"] = {"Potion": 1}
    if "version" not in player or player["version"] < 1:
        player["version"] = 1
    return player

def new_player():
  return {"version": 1, "name": "Hero", "hp": 100, "max_hp": 100,
          "attack": 12, "level": 1, "xp": 0, "gold": 0, "inventory": {"Potion": 2}}

In [ ]:
def make_bar(current, maximum, length=20):
    current = max(0, current)
    filled = int(length * current / maximum) if maximum else 0
    return "[" + "█" * filled + "░" * (length - filled) + f"] {current}/{maximum}"

status_output = widgets.Output()

def render_status(player, enemy=None):
    xp_needed = player["level"] * 25
    with status_output:
        clear_output()
        print(f"{player['name']} — Level {player['level']}")
        print("HP: ", make_bar(player["hp"], player["max_hp"]))
        print("XP: ", make_bar(player["xp"], xp_needed))
        print("Gold:", player["gold"])
        if enemy:
            print(f"\n{enemy['name']} HP:", make_bar(enemy["hp"], enemy["max_hp"]))

In [ ]:
ITEMS = {
    "Potion": {"effect": "heal", "amount": 30, "price": 10},
}

shop_output = widgets.Output()

def render_shop(player):
    with shop_output:
        clear_output()
        print(f"Gold: {player['gold']}")
        for name, item in ITEMS.items():
            print(f"  {name} — {item['price']} gold")

def make_shop_buttons(player):
    buttons = []
    for item_name in ITEMS:
        b = widgets.Button(description=f"Buy {item_name}")
        b.on_click(lambda btn, name=item_name: buy_item(player, name))
        buttons.append(b)
    return widgets.HBox(buttons)

inventory_output = widgets.Output()

def render_inventory(player):
    with inventory_output:
        clear_output()
        if not player["inventory"]:
            print("Inventory is empty.")
        for name, count in player["inventory"].items():
            print(f"  {name} x{count}")

def use_item_ui(player, item_name):
    if item_name not in ITEMS:
        log("That's not a real item.")
        return
    if player["inventory"].get(item_name, 0) <= 0:
        log(f"You don't have any {item_name}.")
        return
    item = ITEMS[item_name]
    if item["effect"] == "heal":
        player["hp"] = min(player["max_hp"], player["hp"] + item["amount"])
        log(f"Used a {item_name}, healed to {player['hp']} HP.")
    player["inventory"][item_name] -= 1
    save_game(player)
    render_inventory(player)
    render_status(player, current_enemy)

def make_item_buttons(player):
    buttons = []
    for item_name in player["inventory"]:
        b = widgets.Button(description=f"Use {item_name}")
        b.on_click(lambda btn, name=item_name: use_item_ui(player, name))
        buttons.append(b)
    return widgets.HBox(buttons)


def buy_item(player, item_name):
    item = ITEMS[item_name]
    if player["gold"] >= item["price"]:
        player["gold"] -= item["price"]
        player["inventory"][item_name] = player["inventory"].get(item_name, 0) + 1
        log(f"Bought a {item_name}.")
        save_game(player)
    else:
        log(f"Not enough gold for {item_name}.")
    render_shop(player)
    render_status(player, current_enemy)


In [ ]:
def save_battle_state(player, enemy):
    save_game({"player": player, "enemy": enemy, "in_battle": True})

ENEMIES = [
    {"name": "Goblin", "hp": 30, "attack": 6, "xp": 10, "gold": 5},
    {"name": "Wolf",   "hp": 25, "attack": 8, "xp": 12, "gold": 4},
    {"name": "Orc",    "hp": 45, "attack": 10, "xp": 20, "gold": 10},
    {"name": "Skeleton","hp": 35, "attack": 7, "xp": 15, "gold": 8},
]

log_output = widgets.Output(layout={'border': '1px solid gray', 'height': '150px', 'overflow_y': 'auto'})

def log(message):
    with log_output:
        print(message)

current_enemy = None  # tracks the active fight

def start_battle(player):
    global current_enemy
    current_enemy = dict(random.choice(ENEMIES))
    current_enemy["max_hp"] = current_enemy["hp"]
    log(f"A wild {current_enemy['name']} appears!")
    render_status(player, current_enemy)

def on_attack_click(b):
    global current_enemy
    if current_enemy is None or current_enemy["hp"] <= 0:
        return
    dmg = random.randint(player["attack"] - 3, player["attack"] + 3)
    current_enemy["hp"] -= dmg
    log(f"You hit the {current_enemy['name']} for {dmg}.")
    if current_enemy["hp"] <= 0:
        log(f"Victory! +{current_enemy['xp']} XP, +{current_enemy['gold']} gold")
        player["xp"] += current_enemy["xp"]
        player["gold"] += current_enemy["gold"]
        check_level_up(player)
        save_game(player)
        current_enemy = None
        render_status(player)
        return
    dmg = random.randint(current_enemy["attack"] - 2, current_enemy["attack"] + 2)
    player["hp"] -= dmg
    log(f"The {current_enemy['name']} hits you for {dmg}.")
    if player["hp"] <= 0:
        log("You were defeated...")
        current_enemy = None
    render_status(player, current_enemy)

attack_button = widgets.Button(description="Attack", button_style="danger")
fight_button  = widgets.Button(description="Find Battle", button_style="success")
rest_button   = widgets.Button(description="Rest", button_style="info")
save_button   = widgets.Button(description="Save")

attack_button.on_click(on_attack_click)
fight_button.on_click(lambda b: start_battle(player))

def on_rest_click(b):
    player["hp"] = player["max_hp"]
    log("You rest and recover all HP.")
    render_status(player, current_enemy)
rest_button.on_click(on_rest_click)

def on_save_click(b):
    save_game(player)
    log("Game saved.")
save_button.on_click(on_save_click)

In [ ]:
def check_level_up(player):
    xp_needed = player["level"] * 25  # each level needs more XP than the last
    while player["xp"] >= xp_needed:
        player["xp"] -= xp_needed
        player["level"] += 1
        player["max_hp"] += 15
        player["hp"] = player["max_hp"]  # full heal on level up
        player["attack"] += 3
        print(f"Level up! You are now level {player['level']}.")
        xp_needed = player["level"] * 25

In [ ]:
def new_game_option(player):
  confirm = input("Are you sure you want to erase and start over? (yes/no)\n> ").strip().lower()
  if confirm == "yes":
        if os.path.exists(SAVE_FILE):
            os.remove(SAVE_FILE)
        print("Save deleted. Starting fresh.")
        return new_player()
  print("Cancelled.")
  return player

In [ ]:
loaded_save_data = load_game()
current_enemy = None

if loaded_save_data and isinstance(loaded_save_data, dict) and "in_battle" in loaded_save_data:
    player = upgrade_save(loaded_save_data["player"])
    current_enemy = loaded_save_data["enemy"]
    log(f"Resumed your fight against {current_enemy['name']}!")
elif loaded_save_data:
    player = upgrade_save(loaded_save_data)
else:
    player = new_player()
    log("Starting a new game.")

In [ ]:
raw = load_game()
print("player value:", raw["player"])
print("enemy value:", raw["enemy"])
print("in_battle value:", raw["in_battle"])

In [ ]:
player = load_game() or new_player()
current_enemy = None
shop_toggle = widgets.Button(description="Shop")

def on_shop_toggle(b):
    render_shop(player)
    display(shop_output, make_shop_buttons(player))

shop_toggle.on_click(on_shop_toggle)

item_toggle = widgets.Button(description="Use Item")

def on_item_toggle(b):
    render_inventory(player)
    display(inventory_output, make_item_buttons(player))

item_toggle.on_click(on_item_toggle)

save_battle_button = widgets.Button(description="Save & Quit Battle", button_style="warning")

def on_save_battle_click(b):
    if current_enemy is not None:
        save_game({"player": player, "enemy": current_enemy, "in_battle": True})
        log("Battle saved. Safe to close the notebook.")
    else:
        log("No battle in progress.")
save_battle_button.on_click(on_save_battle_click)

buttons = widgets.HBox([fight_button, attack_button, rest_button, save_button, shop_toggle, item_toggle, save_battle_button])
ui = widgets.VBox([status_output, buttons, log_output, shop_output, inventory_output])


render_status(player, current_enemy)
display(ui)

In [ ]:
def shop(player):
    print(f"Gold: {player['gold']}")
    for name, item in ITEMS.items():
        print(f"  {name} - {item['price']} gold")
    choice = input("Buy which item? (or 'exit')\n> ").strip()
    if choice in ITEMS and player["gold"] >= ITEMS[choice]["price"]:
        player["gold"] -= ITEMS[choice]["price"]
        player["inventory"][choice] = player["inventory"].get(choice, 0) + 1
        print(f"Bought a {choice}.")
        save_game(player)
    elif choice != "exit":
        print("Can't buy that right now.")





def battle(player, enemy=None):
  if enemy is None:
    enemy = dict(random.choice(ENEMIES))
  print(f"\nA wild {enemy[ 'name' ]} appears! ")
  while enemy[ "hp" ] > 0 and player[ "hp" ] > 0:
    choice = input("1) Attack  2) Save & Quit\n> ").strip()
    if choice == "2":
      save_battle_state(player, enemy)
      print("Battle saved. See you next time!")
      return "saved"
    dmg = random.randint(player["attack"] - 3, player["attack"] + 3)
    enemy["hp"] -= dmg
    print(f"You hit the {enemy['name']} for {dmg}. (Enemy HP: {max(enemy['hp'], 0)})")
    if enemy["hp"] <= 0:
      break
    dmg = random.randint(enemy["attack"] - 2, enemy["attack"] + 2)
    player["hp"] -= dmg
    print(f"The {enemy['name']} hits you for {dmg}. (Your HP: {max(player['hp'], 0)})")
  if player["hp"] > 0:
    print(f"Victory! +{enemy['xp']} XP, +{enemy['gold']} gold")
    player["xp"] += enemy["xp"]
    player["gold"] += enemy["gold"]
    check_level_up(player)
    save_game(player)
    return "won"
  else:
    print("You were defeated...")
    return "lost"

In [ ]:
def main():
  loaded_save_data = load_game()
  player = None # Initialize player
  in_battle_state = False
  enemy_from_save = None

  if loaded_save_data:
    # Check if the loaded data is a battle state dictionary
    if isinstance(loaded_save_data, dict) and "player" in loaded_save_data and "enemy" in loaded_save_data and "in_battle" in loaded_save_data:
      player = loaded_save_data["player"]
      in_battle_state = loaded_save_data["in_battle"]
      enemy_from_save = loaded_save_data["enemy"]
      print(f"Loaded game from battle state with {enemy_from_save['name']}.")
    else:
      # Assume it's a direct player save
      player = loaded_save_data

    player = upgrade_save(player)
    print(f"Welcome back, {player['name']}! (Level{player['level']})")
  else:
    player = new_player()
    print("Starting new game. ")

  # If we loaded from a battle state, resume the battle
  if in_battle_state and player and enemy_from_save:
      print(f"\nResuming battle against {enemy_from_save['name']}...")
      battle_result = battle(player, enemy_from_save)
      if battle_result == "saved":
        return
      if battle_result == "lost":
        player = new_player() # Player lost after resuming battle
      # If battle_result was 'won' or 'saved', the loop will continue normally
      # If saved, the battle function handled the break from battle.
      # If won, player data is already updated.

  while True:
    choice = input("\n1) Fight 2) Rest 3) Stats 4) Save & Quit 5) New Game 6) Shop 7) Use Item\n> ").strip()
    if choice == "1":
        result = battle(player)
        if result == "saved":
          break
        if player["hp"] <= 0:
          player = new_player()
    elif choice == "2":
      player["hp"] = player["max_hp"]
      print("You rest and recover all HP.")
    elif choice == "3":
      print(player)
    elif choice == "4":
      save_game(player)
      print("Game saved. See you next time!")
      break
    elif choice == "5":
      player = new_game_option(player)
    elif choice == "6":
      shop(player)
    elif choice == "7":
      print("Inventory:", player["inventory"])
      item_choice = input("Use which item?\n> ").strip()
      use_item(player, item_choice)
    else:
      print("Invalid choice. ")

In [ ]:
main()

- add comment to the top of each section to explain what it does
- brainstorm with claude on how to make a nice visuals/ui for the code
